# Chapter 3.10: vLLM API Server — FastAPI & OpenAI Compatibility

This notebook provides a comprehensive deep-dive into how vLLM exposes its inference engine through a FastAPI-based HTTP server that is fully compatible with the OpenAI API specification. We will dissect the server architecture, implement endpoints from scratch, and explore streaming, authentication, and rate limiting.

## Learning Objectives
1. Understand vLLM's FastAPI server architecture and request lifecycle
2. Implement OpenAI-compatible endpoints (`/v1/chat/completions`, `/v1/completions`, `/v1/models`)
3. Master SSE (Server-Sent Events) streaming for token-by-token delivery
4. Implement chat template processing with Jinja2
5. Build custom middleware for auth, rate limiting, and logging

## Prerequisites
- Basic understanding of HTTP APIs and REST
- Familiarity with Python async/await
- Basic knowledge of FastAPI

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hideak1/vllm_study/blob/main/notebooks/part3/chapter_3.10_api_server.ipynb)
[![Download Notebook](https://img.shields.io/badge/Download-Notebook-blue)](https://raw.githubusercontent.com/hideak1/vllm_study/main/notebooks/part3/chapter_3.10_api_server.ipynb)

**How to do the exercises:**
1. **Google Colab (Recommended)**: Click the "Open in Colab" badge above — you get your own copy with free GPU.
2. **Local Jupyter**: Clone the repo, run `./start.sh`, then open notebooks at `http://localhost:8888`.
3. **Exercises**: Look for cells marked with 🏋️ **Exercise** or **Assignment**. Fill in the `TODO` sections and run the cell to check your work.

> **Tip**: In Colab, the notebook is automatically copied to your Drive — your changes are saved there.

In [ ]:
# Setup and imports
# !pip install fastapi uvicorn httpx jinja2 pydantic

import json
import time
import uuid
import asyncio
from typing import List, Optional, Dict, Any, Union, AsyncGenerator
from dataclasses import dataclass, field, asdict
from enum import Enum
import hashlib
from collections import defaultdict

# We'll build our own minimal server components
# In production vLLM, these come from fastapi, but we simulate them here
print("All imports successful.")
print("Note: This notebook builds server components as pure Python classes.")
print("To run an actual server, use the vLLM CLI: python -m vllm.entrypoints.openai.api_server")

---
## Part 1: vLLM Server Architecture Overview

The vLLM API server has these key components:

```
Client Request (HTTP)
    │
    ▼
┌─────────────────────────────────────┐
│  FastAPI Application                │
│  ├── Middleware (CORS, Auth, etc.)   │
│  ├── /v1/models                     │
│  ├── /v1/completions                │
│  ├── /v1/chat/completions           │
│  └── /health, /version              │
│                                     │
│  Request Validation (Pydantic)       │
│          │                          │
│          ▼                          │
│  Chat Template Processing (Jinja2)   │
│          │                          │
│          ▼                          │
│  Tokenization                        │
│          │                          │
│          ▼                          │
│  AsyncLLMEngine.generate()           │
│          │                          │
│          ▼                          │
│  Response Formatting (OpenAI spec)   │
│          │                          │
│          ▼                          │
│  SSE Stream or JSON Response         │
└─────────────────────────────────────┘
```

### Key source files in vLLM:
- `vllm/entrypoints/openai/api_server.py` — Main FastAPI app and route definitions
- `vllm/entrypoints/openai/serving_chat.py` — Chat completion handler
- `vllm/entrypoints/openai/serving_completion.py` — Text completion handler
- `vllm/entrypoints/openai/protocol.py` — Pydantic models for request/response
- `vllm/entrypoints/openai/serving_engine.py` — Base class for serving

In [ ]:
# Part 1: OpenAI-compatible data models (mirrors vllm/entrypoints/openai/protocol.py)

from pydantic import BaseModel, Field

class ChatMessage(BaseModel):
    """A single message in a chat conversation."""
    role: str  # 'system', 'user', 'assistant', 'tool'
    content: Optional[str] = None
    name: Optional[str] = None


class ChatCompletionRequest(BaseModel):
    """OpenAI-compatible chat completion request.
    
    See: https://platform.openai.com/docs/api-reference/chat/create
    This mirrors vLLM's ChatCompletionRequest in protocol.py
    """
    model: str
    messages: List[ChatMessage]
    temperature: Optional[float] = 1.0
    top_p: Optional[float] = 1.0
    top_k: Optional[int] = -1
    n: Optional[int] = 1
    max_tokens: Optional[int] = None
    stream: Optional[bool] = False
    stop: Optional[Union[str, List[str]]] = None
    presence_penalty: Optional[float] = 0.0
    frequency_penalty: Optional[float] = 0.0
    user: Optional[str] = None
    seed: Optional[int] = None
    # vLLM extensions
    best_of: Optional[int] = None
    repetition_penalty: Optional[float] = 1.0
    min_p: Optional[float] = 0.0
    guided_json: Optional[str] = None
    guided_regex: Optional[str] = None


class CompletionRequest(BaseModel):
    """OpenAI-compatible text completion request."""
    model: str
    prompt: Union[str, List[str], List[int], List[List[int]]]
    temperature: Optional[float] = 1.0
    top_p: Optional[float] = 1.0
    n: Optional[int] = 1
    max_tokens: Optional[int] = 16
    stream: Optional[bool] = False
    stop: Optional[Union[str, List[str]]] = None
    presence_penalty: Optional[float] = 0.0
    frequency_penalty: Optional[float] = 0.0
    echo: Optional[bool] = False
    logprobs: Optional[int] = None


class UsageInfo(BaseModel):
    """Token usage statistics."""
    prompt_tokens: int
    completion_tokens: int
    total_tokens: int


class ChatCompletionChoice(BaseModel):
    index: int
    message: ChatMessage
    finish_reason: Optional[str] = None


class ChatCompletionResponse(BaseModel):
    """OpenAI-compatible chat completion response."""
    id: str
    object: str = "chat.completion"
    created: int
    model: str
    choices: List[ChatCompletionChoice]
    usage: UsageInfo


# Demo: create a request
req = ChatCompletionRequest(
    model="meta-llama/Llama-3-8B-Instruct",
    messages=[
        ChatMessage(role="system", content="You are a helpful assistant."),
        ChatMessage(role="user", content="Explain PagedAttention in 2 sentences.")
    ],
    temperature=0.7,
    max_tokens=100,
    stream=False
)

print("=== Chat Completion Request ===")
print(json.dumps(req.model_dump(), indent=2))

In [ ]:
# Demo: create a response
def create_chat_response(
    request: ChatCompletionRequest,
    generated_text: str,
    prompt_tokens: int,
    completion_tokens: int,
    finish_reason: str = "stop"
) -> ChatCompletionResponse:
    """Format a generated text into an OpenAI-compatible response.
    
    This mirrors the logic in vLLM's serving_chat.py
    """
    return ChatCompletionResponse(
        id=f"chatcmpl-{uuid.uuid4().hex[:12]}",
        created=int(time.time()),
        model=request.model,
        choices=[
            ChatCompletionChoice(
                index=0,
                message=ChatMessage(role="assistant", content=generated_text),
                finish_reason=finish_reason
            )
        ],
        usage=UsageInfo(
            prompt_tokens=prompt_tokens,
            completion_tokens=completion_tokens,
            total_tokens=prompt_tokens + completion_tokens
        )
    )


response = create_chat_response(
    req,
    generated_text="PagedAttention manages KV cache memory using virtual memory paging, "
                   "dividing cache into fixed-size blocks. This eliminates memory waste "
                   "from fragmentation and enables efficient memory sharing across requests.",
    prompt_tokens=25,
    completion_tokens=35
)

print("=== Chat Completion Response ===")
print(json.dumps(response.model_dump(), indent=2))

---
## Part 2: Request Validation and Parameter Handling

vLLM validates incoming requests and converts OpenAI-style parameters to its internal `SamplingParams`. This involves:
1. Checking parameter ranges
2. Converting between OpenAI conventions and vLLM conventions
3. Handling default values
4. Validating model name against loaded models

In [ ]:
class RequestValidator:
    """Validates and converts API requests to internal parameters.
    
    Mirrors logic in vllm/entrypoints/openai/serving_engine.py
    """
    
    # OpenAI parameter ranges
    VALID_RANGES = {
        'temperature': (0.0, 2.0),
        'top_p': (0.0, 1.0),
        'n': (1, 128),
        'presence_penalty': (-2.0, 2.0),
        'frequency_penalty': (-2.0, 2.0),
    }
    
    def __init__(self, available_models: List[str], max_model_len: int = 4096):
        self.available_models = set(available_models)
        self.max_model_len = max_model_len
    
    def validate_chat_request(self, request: ChatCompletionRequest) -> List[str]:
        """Validate a chat completion request. Returns list of error messages."""
        errors = []
        
        # Check model exists
        if request.model not in self.available_models:
            errors.append(f"Model '{request.model}' not found. Available: {self.available_models}")
        
        # Check messages not empty
        if not request.messages:
            errors.append("'messages' must not be empty")
        
        # Check parameter ranges
        for param, (low, high) in self.VALID_RANGES.items():
            value = getattr(request, param, None)
            if value is not None and (value < low or value > high):
                errors.append(f"'{param}' must be between {low} and {high}, got {value}")
        
        # Check max_tokens
        if request.max_tokens is not None:
            if request.max_tokens < 1:
                errors.append(f"'max_tokens' must be >= 1, got {request.max_tokens}")
            if request.max_tokens > self.max_model_len:
                errors.append(f"'max_tokens' ({request.max_tokens}) exceeds model max length ({self.max_model_len})")
        
        # Check best_of >= n
        if request.best_of is not None and request.n is not None:
            if request.best_of < request.n:
                errors.append(f"'best_of' ({request.best_of}) must be >= 'n' ({request.n})")
        
        # Check streaming compatibility
        if request.stream and request.best_of and request.best_of > 1:
            errors.append("'best_of' > 1 is not supported with streaming")
        
        return errors
    
    def to_sampling_params(self, request: ChatCompletionRequest) -> dict:
        """Convert an API request to internal SamplingParams.
        
        This is where OpenAI-style parameters map to vLLM-style parameters.
        """
        return {
            'n': request.n or 1,
            'best_of': request.best_of,
            'presence_penalty': request.presence_penalty or 0.0,
            'frequency_penalty': request.frequency_penalty or 0.0,
            'repetition_penalty': request.repetition_penalty or 1.0,
            'temperature': request.temperature if request.temperature is not None else 1.0,
            'top_p': request.top_p if request.top_p is not None else 1.0,
            'top_k': request.top_k if request.top_k is not None else -1,
            'min_p': request.min_p or 0.0,
            'max_tokens': request.max_tokens or self.max_model_len,
            'stop': [request.stop] if isinstance(request.stop, str) else (request.stop or []),
            'seed': request.seed,
        }


# Test validation
validator = RequestValidator(
    available_models=["meta-llama/Llama-3-8B-Instruct", "mistralai/Mistral-7B-v0.1"],
    max_model_len=8192
)

# Valid request
errors = validator.validate_chat_request(req)
print(f"Valid request errors: {errors}")

# Invalid request
bad_req = ChatCompletionRequest(
    model="nonexistent-model",
    messages=[],
    temperature=5.0,
    presence_penalty=-3.0,
    max_tokens=100000
)
errors = validator.validate_chat_request(bad_req)
print(f"\nInvalid request errors ({len(errors)}):")
for e in errors:
    print(f"  - {e}")

# Parameter conversion
print(f"\n=== Converted SamplingParams ===")
sp = validator.to_sampling_params(req)
for k, v in sp.items():
    print(f"  {k}: {v}")

---
## Part 3: Chat Template Processing (Jinja2)

Different models require different prompt formats. vLLM uses **Jinja2 templates** (stored in the model's `tokenizer_config.json`) to convert a list of chat messages into the model-specific prompt format.

For example:
- **Llama 3**: `<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n{content}<|eot_id|>...`
- **ChatML**: `<|im_start|>system\n{content}<|im_end|>\n...`
- **Mistral**: `[INST] {content} [/INST]`

In [ ]:
from jinja2 import Template, Environment, BaseLoader

# Common chat templates used by popular models
CHAT_TEMPLATES = {
    "llama3": (
        "{% for message in messages %}"
        "<|start_header_id|>{{ message.role }}<|end_header_id|>\n\n"
        "{{ message.content }}<|eot_id|>"
        "{% endfor %}"
        "{% if add_generation_prompt %}"
        "<|start_header_id|>assistant<|end_header_id|>\n\n"
        "{% endif %}"
    ),
    "chatml": (
        "{% for message in messages %}"
        "<|im_start|>{{ message.role }}\n"
        "{{ message.content }}<|im_end|>\n"
        "{% endfor %}"
        "{% if add_generation_prompt %}"
        "<|im_start|>assistant\n"
        "{% endif %}"
    ),
    "mistral": (
        "{{ bos_token }}"
        "{% for message in messages %}"
        "{% if message.role == 'user' %}"
        "[INST] {{ message.content }} [/INST]"
        "{% elif message.role == 'assistant' %}"
        " {{ message.content }}</s>"
        "{% endif %}"
        "{% endfor %}"
    ),
    "alpaca": (
        "{% for message in messages %}"
        "{% if message.role == 'system' %}"
        "### Instruction:\n{{ message.content }}\n\n"
        "{% elif message.role == 'user' %}"
        "### Input:\n{{ message.content }}\n\n"
        "{% elif message.role == 'assistant' %}"
        "### Response:\n{{ message.content }}\n\n"
        "{% endif %}"
        "{% endfor %}"
        "{% if add_generation_prompt %}"
        "### Response:\n"
        "{% endif %}"
    )
}


class ChatTemplateProcessor:
    """Process chat messages using Jinja2 templates.
    
    Mirrors vLLM's chat template handling in serving_chat.py
    """
    def __init__(self, template_str: str):
        self.env = Environment(loader=BaseLoader())
        self.template = self.env.from_string(template_str)
    
    def apply(
        self,
        messages: List[ChatMessage],
        add_generation_prompt: bool = True,
        bos_token: str = "<s>",
        eos_token: str = "</s>"
    ) -> str:
        """Apply the chat template to format messages into a prompt string."""
        msg_dicts = [m.model_dump() for m in messages]
        return self.template.render(
            messages=msg_dicts,
            add_generation_prompt=add_generation_prompt,
            bos_token=bos_token,
            eos_token=eos_token
        )


# Demo: apply different templates to the same messages
messages = [
    ChatMessage(role="system", content="You are a helpful coding assistant."),
    ChatMessage(role="user", content="Write a Python function to sort a list."),
]

print("=== Same messages, different templates ===")
for name, template_str in CHAT_TEMPLATES.items():
    processor = ChatTemplateProcessor(template_str)
    result = processor.apply(messages)
    print(f"\n--- {name.upper()} ---")
    print(result)
    print(f"Length: {len(result)} chars")

In [ ]:
# Multi-turn conversation template rendering
multi_turn = [
    ChatMessage(role="system", content="You are a helpful assistant."),
    ChatMessage(role="user", content="What is vLLM?"),
    ChatMessage(role="assistant", content="vLLM is a high-throughput LLM serving engine that uses PagedAttention."),
    ChatMessage(role="user", content="How does PagedAttention work?"),
]

processor = ChatTemplateProcessor(CHAT_TEMPLATES["llama3"])
prompt = processor.apply(multi_turn)
print("=== Multi-turn Conversation (Llama 3 format) ===")
print(prompt)
print(f"\nTotal prompt length: {len(prompt)} chars")
print(f"Number of turns: {len(multi_turn)}")

# Show structure
print("\n=== Template Structure Analysis ===")
for i, msg in enumerate(multi_turn):
    print(f"  Turn {i}: role={msg.role}, content_len={len(msg.content or '')}")

---
## Part 4: Token Counting and Usage Reporting

The OpenAI API returns usage statistics with every response. vLLM must track:
- `prompt_tokens`: number of tokens in the input
- `completion_tokens`: number of tokens generated
- `total_tokens`: sum of both

In [ ]:
class SimpleTokenizer:
    """A very simple whitespace-based tokenizer for demonstration.
    
    In vLLM, the actual model tokenizer (e.g., SentencePiece, tiktoken) is used.
    """
    def __init__(self, vocab_size: int = 32000):
        self.vocab_size = vocab_size
    
    def encode(self, text: str) -> List[int]:
        """Tokenize text into token IDs (simplified)."""
        words = text.split()
        # Use hash to get pseudo-consistent token IDs
        return [hash(w) % self.vocab_size for w in words]
    
    def decode(self, token_ids: List[int]) -> str:
        """Decode token IDs back to text (simplified)."""
        return " ".join(f"<tok_{tid}>" for tid in token_ids)
    
    def count_tokens(self, text: str) -> int:
        """Count the number of tokens in text."""
        return len(self.encode(text))


class UsageTracker:
    """Track token usage across requests.
    
    In production, this is used for billing, rate limiting, and monitoring.
    """
    def __init__(self):
        self.total_prompt_tokens = 0
        self.total_completion_tokens = 0
        self.request_count = 0
        self.per_model_usage = defaultdict(lambda: {'prompt': 0, 'completion': 0, 'requests': 0})
    
    def record(self, model: str, prompt_tokens: int, completion_tokens: int):
        self.total_prompt_tokens += prompt_tokens
        self.total_completion_tokens += completion_tokens
        self.request_count += 1
        self.per_model_usage[model]['prompt'] += prompt_tokens
        self.per_model_usage[model]['completion'] += completion_tokens
        self.per_model_usage[model]['requests'] += 1
    
    def summary(self) -> dict:
        return {
            'total_requests': self.request_count,
            'total_prompt_tokens': self.total_prompt_tokens,
            'total_completion_tokens': self.total_completion_tokens,
            'total_tokens': self.total_prompt_tokens + self.total_completion_tokens,
            'per_model': dict(self.per_model_usage)
        }


# Demo
tokenizer = SimpleTokenizer()
tracker = UsageTracker()

# Simulate several requests
simulated_requests = [
    ("meta-llama/Llama-3-8B", "Explain how vLLM handles KV cache management using paged attention", 
     "vLLM uses PagedAttention to manage KV cache as virtual memory pages, eliminating fragmentation."),
    ("meta-llama/Llama-3-8B", "What is tensor parallelism?",
     "Tensor parallelism splits model layers across GPUs, each computing a portion of matrix operations."),
    ("mistralai/Mistral-7B", "Write a hello world program",
     "print('Hello, World!')"),
]

for model, prompt, completion in simulated_requests:
    prompt_tokens = tokenizer.count_tokens(prompt)
    completion_tokens = tokenizer.count_tokens(completion)
    tracker.record(model, prompt_tokens, completion_tokens)
    print(f"Request to {model}: prompt={prompt_tokens} tokens, completion={completion_tokens} tokens")

print(f"\n=== Usage Summary ===")
summary = tracker.summary()
print(json.dumps(summary, indent=2))

---
## Part 5: SSE Streaming Implementation

Streaming delivers tokens as they are generated, using **Server-Sent Events (SSE)**. The format follows the OpenAI spec:

```
data: {"id":"chatcmpl-xxx","choices":[{"delta":{"content":" token"},"index":0}]}

data: {"id":"chatcmpl-xxx","choices":[{"delta":{"content":" next"},"index":0}]}

data: [DONE]
```

Each chunk contains a `delta` (incremental content) instead of the full `message`.

In [ ]:
class StreamChunk(BaseModel):
    """A single SSE chunk for streaming responses."""
    id: str
    object: str = "chat.completion.chunk"
    created: int
    model: str
    choices: List[dict]


class SSEFormatter:
    """Format responses as Server-Sent Events.
    
    Mirrors vLLM's SSE formatting in the streaming response handler.
    """
    @staticmethod
    def format_chunk(data: dict) -> str:
        """Format a single SSE data line."""
        return f"data: {json.dumps(data)}\n\n"
    
    @staticmethod
    def format_done() -> str:
        """Format the [DONE] sentinel."""
        return "data: [DONE]\n\n"


async def generate_stream_chunks(
    request: ChatCompletionRequest,
    generated_text: str,
    tokens_per_chunk: int = 1
) -> AsyncGenerator[str, None]:
    """Simulate streaming token generation.
    
    In vLLM, this is an async generator that yields chunks as the
    LLMEngine produces tokens. Here we simulate by splitting text.
    """
    request_id = f"chatcmpl-{uuid.uuid4().hex[:12]}"
    created = int(time.time())
    words = generated_text.split()
    
    # First chunk: role only (no content)
    first_chunk = {
        "id": request_id,
        "object": "chat.completion.chunk",
        "created": created,
        "model": request.model,
        "choices": [{
            "index": 0,
            "delta": {"role": "assistant", "content": ""},
            "finish_reason": None
        }]
    }
    yield SSEFormatter.format_chunk(first_chunk)
    
    # Content chunks
    for i in range(0, len(words), tokens_per_chunk):
        chunk_words = words[i:i + tokens_per_chunk]
        content = " " + " ".join(chunk_words)
        
        chunk = {
            "id": request_id,
            "object": "chat.completion.chunk",
            "created": created,
            "model": request.model,
            "choices": [{
                "index": 0,
                "delta": {"content": content},
                "finish_reason": None
            }]
        }
        yield SSEFormatter.format_chunk(chunk)
        await asyncio.sleep(0.05)  # Simulate generation time
    
    # Final chunk with finish_reason
    final_chunk = {
        "id": request_id,
        "object": "chat.completion.chunk",
        "created": created,
        "model": request.model,
        "choices": [{
            "index": 0,
            "delta": {},
            "finish_reason": "stop"
        }]
    }
    yield SSEFormatter.format_chunk(final_chunk)
    yield SSEFormatter.format_done()


# Demo: Simulate streaming
async def demo_streaming():
    generated = ("PagedAttention manages KV cache memory using virtual memory paging. "
                 "It divides cache into fixed-size blocks that can be allocated "
                 "and freed independently, eliminating fragmentation.")
    
    print("=== Simulated SSE Stream ===")
    print("(Each line below is a separate SSE event)\n")
    
    collected_text = ""
    chunk_count = 0
    async for sse_line in generate_stream_chunks(req, generated, tokens_per_chunk=2):
        chunk_count += 1
        print(sse_line.strip())
        # Parse back to extract content
        if sse_line.startswith("data: {"):
            data = json.loads(sse_line[6:])
            delta = data["choices"][0]["delta"]
            if "content" in delta:
                collected_text += delta["content"]
    
    print(f"\n=== Reconstructed from stream ({chunk_count} chunks) ===")
    print(collected_text.strip())

await demo_streaming()

---
## Part 6: Error Handling and Status Codes

The OpenAI API uses specific error formats and HTTP status codes. vLLM mirrors these for compatibility.

In [ ]:
class APIError:
    """OpenAI-compatible API error responses.
    
    Error format:
    {
      "error": {
        "message": "...",
        "type": "...",
        "param": "...",
        "code": "..."
      }
    }
    """
    
    @staticmethod
    def create_error(message: str, error_type: str, param: str = None, 
                     code: str = None, http_status: int = 400) -> dict:
        return {
            "status_code": http_status,
            "body": {
                "error": {
                    "message": message,
                    "type": error_type,
                    "param": param,
                    "code": code
                }
            }
        }
    
    @staticmethod
    def model_not_found(model: str) -> dict:
        return APIError.create_error(
            f"The model `{model}` does not exist.",
            "invalid_request_error", "model", "model_not_found", 404
        )
    
    @staticmethod
    def invalid_parameter(param: str, message: str) -> dict:
        return APIError.create_error(
            message, "invalid_request_error", param, None, 400
        )
    
    @staticmethod
    def context_length_exceeded(prompt_tokens: int, max_tokens: int, model_max: int) -> dict:
        return APIError.create_error(
            f"This model's maximum context length is {model_max} tokens. "
            f"However, your messages resulted in {prompt_tokens} tokens, "
            f"and you requested {max_tokens} max completion tokens, "
            f"totaling {prompt_tokens + max_tokens} tokens.",
            "invalid_request_error", None, "context_length_exceeded", 400
        )
    
    @staticmethod
    def rate_limit_exceeded(limit: int, window: str) -> dict:
        return APIError.create_error(
            f"Rate limit exceeded: {limit} requests per {window}.",
            "rate_limit_error", None, "rate_limit_exceeded", 429
        )
    
    @staticmethod
    def server_error(message: str = "Internal server error") -> dict:
        return APIError.create_error(
            message, "server_error", None, None, 500
        )


# Demo: various error types
errors = [
    ("Model not found", APIError.model_not_found("gpt-5-turbo")),
    ("Invalid param", APIError.invalid_parameter("temperature", "temperature must be between 0 and 2")),
    ("Context exceeded", APIError.context_length_exceeded(7500, 2000, 8192)),
    ("Rate limited", APIError.rate_limit_exceeded(100, "minute")),
    ("Server error", APIError.server_error("CUDA out of memory")),
]

for name, error in errors:
    print(f"--- {name} (HTTP {error['status_code']}) ---")
    print(json.dumps(error['body'], indent=2))
    print()

---
## Part 7: Building a Minimal OpenAI-Compatible Server

Let's build a complete minimal server that handles all three main endpoints. We'll implement this as a class that processes requests synchronously (in production, vLLM uses async).

In [ ]:
class ModelInfo(BaseModel):
    """Model information for /v1/models endpoint."""
    id: str
    object: str = "model"
    created: int = 0
    owned_by: str = "vllm"
    max_model_len: int = 4096


class MinimalOpenAIServer:
    """A minimal OpenAI-compatible API server.
    
    Demonstrates the core logic without actual model inference.
    In production vLLM, this is spread across multiple files:
    - api_server.py: route definitions
    - serving_chat.py: chat completion handling
    - serving_completion.py: text completion handling
    """
    
    def __init__(self, models: List[ModelInfo]):
        self.models = {m.id: m for m in models}
        self.tokenizer = SimpleTokenizer()
        self.usage_tracker = UsageTracker()
        self.template_processor = ChatTemplateProcessor(CHAT_TEMPLATES["chatml"])
        self.validator = RequestValidator(
            list(self.models.keys()),
            max_model_len=max(m.max_model_len for m in models)
        )
    
    def list_models(self) -> dict:
        """GET /v1/models"""
        return {
            "object": "list",
            "data": [m.model_dump() for m in self.models.values()]
        }
    
    def _simulate_generation(self, prompt: str, max_tokens: int) -> str:
        """Simulate text generation (placeholder for real LLM inference)."""
        responses = [
            "PagedAttention divides KV cache into blocks like virtual memory pages.",
            "This eliminates memory fragmentation and enables efficient sharing.",
            "The scheduler manages request batching for maximum GPU utilization.",
            "Continuous batching allows new requests to join ongoing batches.",
        ]
        idx = hash(prompt) % len(responses)
        return responses[idx]
    
    def chat_completion(self, request: ChatCompletionRequest) -> Union[dict, str]:
        """POST /v1/chat/completions"""
        # Step 1: Validate
        errors = self.validator.validate_chat_request(request)
        if errors:
            return APIError.invalid_parameter(None, "; ".join(errors))
        
        # Step 2: Apply chat template
        prompt = self.template_processor.apply(request.messages)
        prompt_tokens = self.tokenizer.count_tokens(prompt)
        
        # Step 3: Check context length
        model_info = self.models[request.model]
        max_tokens = request.max_tokens or (model_info.max_model_len - prompt_tokens)
        if prompt_tokens + max_tokens > model_info.max_model_len:
            return APIError.context_length_exceeded(
                prompt_tokens, max_tokens, model_info.max_model_len
            )
        
        # Step 4: Generate
        generated_text = self._simulate_generation(prompt, max_tokens)
        completion_tokens = self.tokenizer.count_tokens(generated_text)
        
        # Step 5: Track usage
        self.usage_tracker.record(request.model, prompt_tokens, completion_tokens)
        
        # Step 6: Format response
        response = create_chat_response(
            request, generated_text, prompt_tokens, completion_tokens
        )
        return response.model_dump()
    
    def text_completion(self, request: CompletionRequest) -> dict:
        """POST /v1/completions"""
        prompt_text = request.prompt if isinstance(request.prompt, str) else str(request.prompt)
        prompt_tokens = self.tokenizer.count_tokens(prompt_text)
        
        generated = self._simulate_generation(prompt_text, request.max_tokens or 16)
        completion_tokens = self.tokenizer.count_tokens(generated)
        
        self.usage_tracker.record(request.model, prompt_tokens, completion_tokens)
        
        return {
            "id": f"cmpl-{uuid.uuid4().hex[:12]}",
            "object": "text_completion",
            "created": int(time.time()),
            "model": request.model,
            "choices": [{
                "text": generated,
                "index": 0,
                "logprobs": None,
                "finish_reason": "stop"
            }],
            "usage": {
                "prompt_tokens": prompt_tokens,
                "completion_tokens": completion_tokens,
                "total_tokens": prompt_tokens + completion_tokens
            }
        }


# Create server
server = MinimalOpenAIServer([
    ModelInfo(id="meta-llama/Llama-3-8B-Instruct", max_model_len=8192),
    ModelInfo(id="mistralai/Mistral-7B-v0.1", max_model_len=4096),
])

# Test endpoints
print("=== GET /v1/models ===")
print(json.dumps(server.list_models(), indent=2))

In [ ]:
# Test chat completion endpoint
print("=== POST /v1/chat/completions ===")
chat_req = ChatCompletionRequest(
    model="meta-llama/Llama-3-8B-Instruct",
    messages=[
        ChatMessage(role="user", content="What is PagedAttention?")
    ],
    temperature=0.7,
    max_tokens=200
)
result = server.chat_completion(chat_req)
print(json.dumps(result, indent=2))

# Test with invalid model
print("\n=== Error: Invalid model ===")
bad_req = ChatCompletionRequest(
    model="nonexistent",
    messages=[ChatMessage(role="user", content="test")]
)
print(json.dumps(server.chat_completion(bad_req), indent=2))

# Test text completion
print("\n=== POST /v1/completions ===")
comp_req = CompletionRequest(
    model="meta-llama/Llama-3-8B-Instruct",
    prompt="The key innovation of vLLM is",
    max_tokens=50
)
print(json.dumps(server.text_completion(comp_req), indent=2))

---
## Part 8: Middleware — Authentication, Rate Limiting, Logging

In production, API servers need middleware for security and observability. vLLM supports custom middleware through FastAPI's middleware system.

In [ ]:
class AuthMiddleware:
    """API key authentication middleware.
    
    In vLLM, this can be configured via --api-key flag.
    """
    def __init__(self, valid_api_keys: List[str]):
        self.valid_keys = set(valid_api_keys)
    
    def authenticate(self, api_key: Optional[str]) -> tuple:
        """Returns (is_valid, error_or_None)."""
        if not self.valid_keys:
            return True, None  # No auth configured
        
        if not api_key:
            return False, APIError.create_error(
                "Missing API key. Include 'Authorization: Bearer <key>' header.",
                "authentication_error", None, "missing_api_key", 401
            )
        
        # Strip 'Bearer ' prefix if present
        key = api_key.replace("Bearer ", "").strip()
        if key not in self.valid_keys:
            return False, APIError.create_error(
                "Invalid API key.",
                "authentication_error", None, "invalid_api_key", 401
            )
        
        return True, None


class TokenBucketRateLimiter:
    """Token bucket rate limiter.
    
    Each client gets a bucket of tokens. Tokens refill at a steady rate.
    Each request consumes one token. When the bucket is empty, requests are rejected.
    """
    def __init__(self, rate: float = 10.0, burst: int = 20):
        self.rate = rate      # tokens per second
        self.burst = burst    # max bucket size
        self.buckets: Dict[str, dict] = {}  # client_id -> {tokens, last_refill}
    
    def _refill(self, bucket: dict) -> None:
        now = time.time()
        elapsed = now - bucket['last_refill']
        bucket['tokens'] = min(self.burst, bucket['tokens'] + elapsed * self.rate)
        bucket['last_refill'] = now
    
    def allow(self, client_id: str) -> tuple:
        """Check if request is allowed. Returns (allowed, remaining_tokens)."""
        if client_id not in self.buckets:
            self.buckets[client_id] = {'tokens': self.burst, 'last_refill': time.time()}
        
        bucket = self.buckets[client_id]
        self._refill(bucket)
        
        if bucket['tokens'] >= 1.0:
            bucket['tokens'] -= 1.0
            return True, int(bucket['tokens'])
        else:
            return False, 0


class RequestLogger:
    """Log API requests for monitoring and debugging."""
    def __init__(self):
        self.logs: List[dict] = []
    
    def log_request(self, method: str, path: str, client_id: str,
                    status_code: int, latency_ms: float, tokens_used: int = 0):
        entry = {
            'timestamp': time.time(),
            'method': method,
            'path': path,
            'client_id': client_id,
            'status_code': status_code,
            'latency_ms': round(latency_ms, 2),
            'tokens_used': tokens_used
        }
        self.logs.append(entry)
        return entry
    
    def get_stats(self) -> dict:
        if not self.logs:
            return {'total_requests': 0}
        
        latencies = [l['latency_ms'] for l in self.logs]
        status_codes = defaultdict(int)
        for l in self.logs:
            status_codes[l['status_code']] += 1
        
        return {
            'total_requests': len(self.logs),
            'avg_latency_ms': sum(latencies) / len(latencies),
            'p50_latency_ms': sorted(latencies)[len(latencies) // 2],
            'p99_latency_ms': sorted(latencies)[int(len(latencies) * 0.99)],
            'status_codes': dict(status_codes),
            'total_tokens': sum(l['tokens_used'] for l in self.logs)
        }


# Demo: middleware stack
auth = AuthMiddleware(["sk-test-key-123", "sk-prod-key-456"])
limiter = TokenBucketRateLimiter(rate=5.0, burst=10)
logger = RequestLogger()

# Test authentication
print("=== Authentication Tests ===")
tests = [
    ("Bearer sk-test-key-123", "Valid key"),
    ("Bearer sk-wrong-key", "Wrong key"),
    (None, "No key"),
]
for key, desc in tests:
    valid, error = auth.authenticate(key)
    status = "PASS" if valid else f"FAIL ({error['body']['error']['code']})"
    print(f"  {desc:.<30s} {status}")

# Test rate limiting
print("\n=== Rate Limiting Tests ===")
for i in range(15):
    allowed, remaining = limiter.allow("client-1")
    if i < 12 or not allowed:
        print(f"  Request {i+1:2d}: {'ALLOWED' if allowed else 'BLOCKED':>7s} (remaining: {remaining})")

In [ ]:
# Full request pipeline with middleware
class FullPipelineServer:
    """Server with full middleware pipeline."""
    
    def __init__(self, server: MinimalOpenAIServer, 
                 auth: AuthMiddleware, limiter: TokenBucketRateLimiter,
                 logger: RequestLogger):
        self.server = server
        self.auth = auth
        self.limiter = limiter
        self.logger = logger
    
    def handle_request(self, method: str, path: str, 
                       api_key: str = None, body: dict = None,
                       client_id: str = "anonymous") -> dict:
        """Process a request through the full middleware pipeline."""
        start_time = time.time()
        
        # Step 1: Authentication
        valid, error = self.auth.authenticate(api_key)
        if not valid:
            latency = (time.time() - start_time) * 1000
            self.logger.log_request(method, path, client_id, 401, latency)
            return error
        
        # Step 2: Rate limiting
        allowed, remaining = self.limiter.allow(client_id)
        if not allowed:
            latency = (time.time() - start_time) * 1000
            self.logger.log_request(method, path, client_id, 429, latency)
            return APIError.rate_limit_exceeded(10, "second")
        
        # Step 3: Route to handler
        if path == "/v1/models" and method == "GET":
            result = self.server.list_models()
            status = 200
        elif path == "/v1/chat/completions" and method == "POST":
            req = ChatCompletionRequest(**body)
            result = self.server.chat_completion(req)
            status = result.get('status_code', 200) if isinstance(result, dict) and 'status_code' in result else 200
        elif path == "/v1/completions" and method == "POST":
            req = CompletionRequest(**body)
            result = self.server.text_completion(req)
            status = 200
        else:
            result = APIError.create_error("Not found", "not_found", None, None, 404)
            status = 404
        
        # Step 4: Log
        latency = (time.time() - start_time) * 1000
        tokens = result.get('usage', {}).get('total_tokens', 0) if isinstance(result, dict) else 0
        self.logger.log_request(method, path, client_id, status, latency, tokens)
        
        return result


# Create and test the full pipeline
full_server = FullPipelineServer(
    server=server,
    auth=AuthMiddleware(["sk-test-123"]),
    limiter=TokenBucketRateLimiter(rate=100, burst=200),
    logger=RequestLogger()
)

# Test various scenarios
print("=== Full Pipeline Tests ===")

# 1. Valid request
result = full_server.handle_request(
    "POST", "/v1/chat/completions",
    api_key="Bearer sk-test-123",
    body={
        "model": "meta-llama/Llama-3-8B-Instruct",
        "messages": [{"role": "user", "content": "Hello!"}],
        "max_tokens": 50
    },
    client_id="user-1"
)
print("1. Valid request:")
print(f"   Status: 200, Model: {result.get('model', 'N/A')}")
print(f"   Response: {result['choices'][0]['message']['content'][:80]}...")

# 2. Missing auth
result = full_server.handle_request("GET", "/v1/models", client_id="user-2")
print(f"\n2. No auth: Status {result['status_code']}")

# 3. List models
result = full_server.handle_request("GET", "/v1/models", api_key="Bearer sk-test-123")
print(f"\n3. List models: {len(result['data'])} models available")

# Show logs
print(f"\n=== Request Logs ===")
stats = full_server.logger.get_stats()
print(json.dumps(stats, indent=2))

---
## Part 9: Demo — Implementing a FastAPI Server (Actual Runnable Code)

Below is the actual FastAPI code that you would use to run a server. It won't run in a Jupyter notebook, but you can copy it to a `.py` file and run it.

In [ ]:
# This cell shows a complete runnable FastAPI server
# Save this to a file (e.g., mini_server.py) and run with:
#   uvicorn mini_server:app --port 8000

FASTAPI_SERVER_CODE = '''
"""Minimal OpenAI-compatible API server using FastAPI.

Usage:
    pip install fastapi uvicorn
    uvicorn mini_server:app --port 8000
    
Then test with:
    curl http://localhost:8000/v1/models
    curl http://localhost:8000/v1/chat/completions \\
      -H "Content-Type: application/json" \\
      -d '{"model": "test-model", "messages": [{"role": "user", "content": "Hello!"}]}'
"""
import time, uuid, json, asyncio
from typing import List, Optional, Union, AsyncGenerator
from fastapi import FastAPI, Request, HTTPException
from fastapi.responses import StreamingResponse, JSONResponse
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel

app = FastAPI(title="Mini vLLM Server")

# CORS middleware (allows browser access)
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

class ChatMessage(BaseModel):
    role: str
    content: Optional[str] = None

class ChatRequest(BaseModel):
    model: str
    messages: List[ChatMessage]
    temperature: float = 1.0
    max_tokens: Optional[int] = 100
    stream: bool = False

@app.get("/v1/models")
async def list_models():
    return {"object": "list", "data": [{"id": "test-model", "object": "model"}]}

@app.get("/health")
async def health():
    return {"status": "ok"}

async def stream_generator(request_id: str, model: str, text: str):
    """Generate SSE stream chunks."""
    words = text.split()
    for word in words:
        chunk = {
            "id": request_id, "object": "chat.completion.chunk",
            "created": int(time.time()), "model": model,
            "choices": [{"index": 0, "delta": {"content": " " + word}, "finish_reason": None}]
        }
        yield f"data: {json.dumps(chunk)}\\n\\n"
        await asyncio.sleep(0.05)
    # Final chunk
    final = {
        "id": request_id, "object": "chat.completion.chunk",
        "created": int(time.time()), "model": model,
        "choices": [{"index": 0, "delta": {}, "finish_reason": "stop"}]
    }
    yield f"data: {json.dumps(final)}\\n\\n"
    yield "data: [DONE]\\n\\n"

@app.post("/v1/chat/completions")
async def chat_completions(request: ChatRequest):
    request_id = f"chatcmpl-{uuid.uuid4().hex[:12]}"
    generated = "This is a simulated response from the mini vLLM server."
    
    if request.stream:
        return StreamingResponse(
            stream_generator(request_id, request.model, generated),
            media_type="text/event-stream"
        )
    
    return {
        "id": request_id, "object": "chat.completion",
        "created": int(time.time()), "model": request.model,
        "choices": [{"index": 0, "message": {"role": "assistant", "content": generated}, "finish_reason": "stop"}],
        "usage": {"prompt_tokens": 10, "completion_tokens": 12, "total_tokens": 22}
    }
'''

print("=== FastAPI Server Code ===")
print(FASTAPI_SERVER_CODE)
print("\nSave the above to 'mini_server.py' and run with:")
print("  uvicorn mini_server:app --port 8000")
print("\nThen test with:")
print('  curl http://localhost:8000/v1/models')
print('  curl -X POST http://localhost:8000/v1/chat/completions \\')
print('    -H "Content-Type: application/json" \\')
print('    -d \'{"model": "test-model", "messages": [{"role": "user", "content": "Hello!"}]}\'  ')

In [ ]:
# Demo: Simulate client requests to test the server logic
# (We test the Python logic directly since we can't start a server in a notebook)

import time

class MockHTTPClient:
    """Simulates HTTP client calls against our server."""
    
    def __init__(self, server: FullPipelineServer, api_key: str):
        self.server = server
        self.api_key = f"Bearer {api_key}"
    
    def chat(self, model: str, messages: List[dict], **kwargs) -> dict:
        body = {"model": model, "messages": messages, **kwargs}
        return self.server.handle_request(
            "POST", "/v1/chat/completions",
            api_key=self.api_key, body=body, client_id="test-client"
        )
    
    def complete(self, model: str, prompt: str, **kwargs) -> dict:
        body = {"model": model, "prompt": prompt, **kwargs}
        return self.server.handle_request(
            "POST", "/v1/completions",
            api_key=self.api_key, body=body, client_id="test-client"
        )
    
    def list_models(self) -> dict:
        return self.server.handle_request(
            "GET", "/v1/models", api_key=self.api_key, client_id="test-client"
        )


# Create client and make requests
client = MockHTTPClient(full_server, "sk-test-123")

# Multiple chat requests
print("=== Batch of Chat Requests ===")
prompts = [
    "What is vLLM?",
    "Explain continuous batching.",
    "How does KV cache work?",
    "What is tensor parallelism?",
    "Compare vLLM and TGI.",
]

for p in prompts:
    result = client.chat(
        "meta-llama/Llama-3-8B-Instruct",
        [{"role": "user", "content": p}],
        max_tokens=50
    )
    reply = result.get('choices', [{}])[0].get('message', {}).get('content', 'N/A')
    tokens = result.get('usage', {}).get('total_tokens', 0)
    print(f"  Q: {p}")
    print(f"  A: {reply[:80]}...")
    print(f"  Tokens: {tokens}\n")

# Show accumulated stats
print("=== Accumulated Statistics ===")
print(json.dumps(full_server.logger.get_stats(), indent=2))

---
## Part 10: vLLM Server Launch and Configuration

In production, vLLM is launched via the command line. Let's explore the key configuration options.

In [ ]:
# vLLM server launch configurations
launch_configs = {
    "Basic": {
        "command": "python -m vllm.entrypoints.openai.api_server --model meta-llama/Llama-3-8B-Instruct",
        "description": "Simplest launch: serves one model on port 8000"
    },
    "With GPU config": {
        "command": (
            "python -m vllm.entrypoints.openai.api_server "
            "--model meta-llama/Llama-3-70B-Instruct "
            "--tensor-parallel-size 4 "
            "--gpu-memory-utilization 0.9 "
            "--max-model-len 8192"
        ),
        "description": "70B model across 4 GPUs with 90% memory utilization"
    },
    "With auth + port": {
        "command": (
            "python -m vllm.entrypoints.openai.api_server "
            "--model mistralai/Mistral-7B-Instruct-v0.2 "
            "--api-key sk-my-secret-key "
            "--port 8080 "
            "--host 0.0.0.0"
        ),
        "description": "With API key auth, custom port, accessible from network"
    },
    "With quantization": {
        "command": (
            "python -m vllm.entrypoints.openai.api_server "
            "--model TheBloke/Llama-2-13B-GPTQ "
            "--quantization gptq "
            "--dtype float16"
        ),
        "description": "GPTQ quantized model for reduced memory"
    },
    "Production": {
        "command": (
            "python -m vllm.entrypoints.openai.api_server "
            "--model meta-llama/Llama-3-8B-Instruct "
            "--api-key $VLLM_API_KEY "
            "--host 0.0.0.0 --port 8000 "
            "--max-num-seqs 256 "
            "--max-num-batched-tokens 32768 "
            "--enable-prefix-caching "
            "--disable-log-requests"
        ),
        "description": "Production config with prefix caching and high throughput"
    },
}

for name, config in launch_configs.items():
    print(f"=== {name} ===")
    print(f"Description: {config['description']}")
    print(f"Command:")
    # Break long command for readability
    parts = config['command'].split(' --')
    print(f"  {parts[0]}")
    for p in parts[1:]:
        print(f"    --{p}")
    print()

In [ ]:
# Client-side usage examples with curl and Python
print("=== Client Usage Examples ===")
print()
print("1. curl - Non-streaming:")
print('''   curl http://localhost:8000/v1/chat/completions \\''')
print('''     -H "Content-Type: application/json" \\''')
print('''     -H "Authorization: Bearer sk-xxx" \\''')
print('''     -d '{"model": "meta-llama/Llama-3-8B-Instruct",''')
print('''          "messages": [{"role": "user", "content": "Hello!"}],''')
print('''          "temperature": 0.7, "max_tokens": 100}' ''')

print()
print("2. curl - Streaming:")
print('''   curl http://localhost:8000/v1/chat/completions \\''')
print('''     -H "Content-Type: application/json" \\''')
print('''     -d '{"model": "meta-llama/Llama-3-8B-Instruct",''')
print('''          "messages": [{"role": "user", "content": "Hello!"}],''')
print('''          "stream": true}' ''')

print()
print("3. Python - OpenAI SDK (drop-in replacement):")
python_example = '''
from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:8000/v1",
    api_key="sk-xxx"  # or "token-abc123" if no auth
)

# Non-streaming
response = client.chat.completions.create(
    model="meta-llama/Llama-3-8B-Instruct",
    messages=[{"role": "user", "content": "Hello!"}],
    temperature=0.7,
    max_tokens=100
)
print(response.choices[0].message.content)

# Streaming
stream = client.chat.completions.create(
    model="meta-llama/Llama-3-8B-Instruct",
    messages=[{"role": "user", "content": "Hello!"}],
    stream=True
)
for chunk in stream:
    if chunk.choices[0].delta.content:
        print(chunk.choices[0].delta.content, end="")
'''
print(python_example)

---
## Hands-On Assignments

### Assignment 1: Add a Custom API Endpoint for Batch Processing

**Task**: Extend the `MinimalOpenAIServer` with a `/v1/batch` endpoint that accepts multiple prompts and returns all results at once.

Requirements:
1. Accept a list of chat completion requests in a single API call
2. Process them sequentially (simulating batch inference)
3. Return all results with per-request usage statistics and a total summary
4. Include error handling: if one request fails, still process the rest
5. Add a `batch_id` to the response for tracking

In [ ]:
# === ASSIGNMENT 1: Batch Processing Endpoint ===

class BatchRequest(BaseModel):
    """Request containing multiple chat completion requests."""
    requests: List[ChatCompletionRequest]
    # Optional: fail_fast stops processing on first error
    fail_fast: bool = False


class BatchResponse(BaseModel):
    """Response containing all batch results."""
    batch_id: str
    results: List[dict]  # Each is either a success response or error
    summary: dict


def handle_batch(server: MinimalOpenAIServer, batch: BatchRequest) -> dict:
    """
    TODO: Implement batch processing.
    
    For each request in batch.requests:
    1. Try to process it via server.chat_completion()
    2. Collect the result (success or error)
    3. If fail_fast=True and an error occurs, stop and return partial results
    4. Build a summary with total_requests, successful, failed, total_tokens
    """
    # YOUR CODE HERE
    pass


# Test:
# batch = BatchRequest(requests=[
#     ChatCompletionRequest(model="meta-llama/Llama-3-8B-Instruct",
#                           messages=[ChatMessage(role="user", content="Q1")], max_tokens=50),
#     ChatCompletionRequest(model="meta-llama/Llama-3-8B-Instruct",
#                           messages=[ChatMessage(role="user", content="Q2")], max_tokens=50),
#     ChatCompletionRequest(model="invalid-model",
#                           messages=[ChatMessage(role="user", content="Q3")], max_tokens=50),
# ])
# result = handle_batch(server, batch)
# print(json.dumps(result, indent=2))

print("Assignment 1: Implement handle_batch() and test with mixed valid/invalid requests.")

### Assignment 2: Implement Token-Based Rate Limiting

**Task**: Build a rate limiter that limits based on **tokens consumed** (not just request count).

Requirements:
1. Track tokens consumed per client per minute
2. Set configurable limits: `tokens_per_minute`, `requests_per_minute`
3. When a client exceeds their limit, return a 429 error with `Retry-After` header info
4. Implement a sliding window (not fixed windows)
5. Test with multiple simulated clients

In [ ]:
# === ASSIGNMENT 2: Token-Based Rate Limiter ===

class TokenRateLimiter:
    """
    TODO: Implement a rate limiter that tracks token usage.
    
    Features:
    - Sliding window token counting
    - Configurable tokens_per_minute and requests_per_minute
    - Returns remaining quota and retry_after on limit
    """
    def __init__(self, tokens_per_minute: int = 10000, requests_per_minute: int = 60):
        self.tokens_per_minute = tokens_per_minute
        self.requests_per_minute = requests_per_minute
        # YOUR CODE: data structures for tracking
    
    def check_and_consume(self, client_id: str, estimated_tokens: int) -> dict:
        """
        Check if the request is allowed and consume tokens.
        
        Returns:
        {
            'allowed': bool,
            'remaining_tokens': int,
            'remaining_requests': int,
            'retry_after_seconds': float (0 if allowed)
        }
        """
        # YOUR CODE HERE
        pass
    
    def report_actual_usage(self, client_id: str, actual_tokens: int):
        """Update with actual token count after generation completes."""
        # YOUR CODE HERE
        pass


print("Assignment 2: Implement TokenRateLimiter with sliding window.")
print("Test with multiple clients exceeding their quotas.")

### Assignment 3: Build a Request Logging and Analytics System

**Task**: Create a comprehensive request logging system with real-time analytics.

Requirements:
1. Log every request with: timestamp, client, model, tokens, latency, status, error
2. Compute rolling statistics: requests/sec, tokens/sec, error rate, p50/p95/p99 latency
3. Generate per-model and per-client breakdowns
4. Implement a simple anomaly detector (flag requests with unusually high latency)
5. Create a dashboard-like summary output

In [ ]:
# === ASSIGNMENT 3: Request Analytics System ===

class AnalyticsSystem:
    """
    TODO: Implement a comprehensive request analytics system.
    
    Features:
    - Detailed request logging
    - Rolling window statistics
    - Per-model and per-client breakdowns
    - Anomaly detection for latency spikes
    - Dashboard-style summary
    """
    def __init__(self, anomaly_threshold_std: float = 3.0):
        self.anomaly_threshold_std = anomaly_threshold_std
        # YOUR CODE: data structures
    
    def log(self, request_data: dict):
        """Log a request."""
        # YOUR CODE HERE
        pass
    
    def get_rolling_stats(self, window_seconds: float = 60.0) -> dict:
        """Get statistics for the last N seconds."""
        # YOUR CODE HERE
        pass
    
    def get_model_breakdown(self) -> dict:
        """Get per-model statistics."""
        # YOUR CODE HERE
        pass
    
    def get_anomalies(self) -> List[dict]:
        """Get requests flagged as anomalous."""
        # YOUR CODE HERE
        pass
    
    def dashboard(self) -> str:
        """Generate a text-based dashboard summary."""
        # YOUR CODE HERE
        pass


print("Assignment 3: Implement AnalyticsSystem and generate a dashboard.")
print("Feed it simulated request data and show model/client breakdowns.")

---
## Summary

In this chapter, we covered the complete vLLM API server architecture:

| Component | Purpose | Key Files |
|-----------|---------|----------|
| FastAPI App | HTTP routing, middleware | `api_server.py` |
| Protocol Models | Request/response validation | `protocol.py` |
| Chat Handler | Chat template + generation | `serving_chat.py` |
| Completion Handler | Text completion | `serving_completion.py` |
| SSE Streaming | Token-by-token delivery | All serving files |
| Chat Templates | Jinja2 prompt formatting | `tokenizer_config.json` |

### Key Takeaways
1. vLLM's API is a **drop-in replacement** for OpenAI's API — just change the `base_url`
2. Chat templates are crucial — wrong templates produce garbage output
3. SSE streaming follows the exact OpenAI format with `delta` objects
4. Production deployments need auth, rate limiting, and monitoring middleware
5. The server is async throughout — FastAPI + AsyncLLMEngine for high concurrency

In [ ]:
print("=== End of Chapter 3.10 ===")
print("\nKey vLLM server commands:")
print("  Start server:  python -m vllm.entrypoints.openai.api_server --model <model>")
print("  With TP:       ... --tensor-parallel-size 4")
print("  With auth:     ... --api-key <key>")
print("  With caching:  ... --enable-prefix-caching")
print("  Health check:  curl http://localhost:8000/health")
print("  List models:   curl http://localhost:8000/v1/models")